In [ ]:
import pandas as pd
from glob import glob
import numpy as np

In [ ]:
subjects = [s.split('/')[-1] for s in sorted(glob('../data/sub-*'))]

In [ ]:
def load_confounds(subject):
    confounds = sorted(glob(f"../outputs/fmriprep/{subject}/func/*.tsv"))
    dfs = [pd.read_csv(c, sep='\t') for c in confounds]
    return dfs

In [ ]:
def extract_cols(df):
    cols = ['framewise_displacement']
    cols += [c for c in df.columns if 'motion_outlier' in c]
    return df[cols]

In [ ]:
n_outliers = []
framewise_displacement = []
for subj in subjects:
    dfs = load_confounds(subj)
    dfs = [extract_cols(d) for d in dfs]
    for df in dfs:
        # add framewise displacement to a list since we're just going to compute
        # a group metric (median, min, max)
        framewise_displacement += df['framewise_displacement'].values.tolist()
        n_outliers.append(len([c for c in df.columns if 'motion' in c]))

In [ ]:
# split outliers into subjects again
n_outliers_subject = np.split(np.array(n_outliers), len(subjects))

In [ ]:
for subj, out in zip(subjects, n_outliers_subject):
    print(f"{subj}: {out.sum()}")

In [ ]:
dfs = load_confounds('sub-sid000009')
# get number of TRs for each run so we can compute percentage
n_trs = [len(df) for df in dfs]
all_n_trs = np.sum(n_trs)

In [ ]:
for subj, out in zip(subjects, n_outliers_subject):
    print(f"{subj}: {out.sum()/all_n_trs*100:.2f} %")

In [ ]:
percentage_outliers = np.array([out.sum()/all_n_trs*100 for out in n_outliers_subject])

Compute the median percentage of outliers, min, and max

In [ ]:
np.round((np.median(percentage_outliers), np.min(percentage_outliers), np.max(percentage_outliers)), 2)

How many subjects with less than 5% outliers?

In [ ]:
np.sum(percentage_outliers < 5)

Now compute median framewise dispalcement, max, and min

In [ ]:
framewise_displacement_subjects = np.split(np.array(framewise_displacement), len(subjects))
median_fd_subjects = [np.nanmedian(fd) for fd in framewise_displacement_subjects]

In [ ]:
np.round((np.nanmedian(median_fd_subjects), np.nanmin(median_fd_subjects), np.nanmax(median_fd_subjects)), 2)